## Final DenseNet Pipeline

In [ ]:
# import needed libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import imageio.v2 as imageio
import json
from collections import defaultdict
import shutil
import cv2
from matplotlib import pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from PIL import Image
from torch.utils.data import DataLoader
from torch.utils.data import Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## Dataloading
Beginning with the metadata

In [ ]:
# load the metadata from dataset.csv

metadata = pd.read_csv('FracAtlas/dataset.csv')
print("Shape of the dataset", metadata.shape)
# prints in a table format the first 10 entries of the dataset.csv dataset
metadata.head(10)

In [ ]:
# check image type
img = cv2.imread("FracAtlas/images/Fractured/IMG0000019.jpg")
print(img.shape)

## Median Frequency Balancing

In [ ]:
# median frequency balancing between all images (fractured and non-fractured)

label = [ 'hand', 'leg','hip','shoulder','mixed']

def estimate_weights_mfb(label):
    class_weights = np.zeros_like(label, dtype=float)
    counts = np.zeros_like(label)
    for i,l in enumerate(label):
        counts[i] = metadata[metadata[l] == 1].shape[0]
    counts = counts.astype(float)
    median_freq = np.median(counts)
    for i, label in enumerate(label):
        class_weights[i] = median_freq / counts[i]
    return class_weights

classweight= estimate_weights_mfb(label)

for i in range(len(label)):
    print(label[i],":", classweight[i])

## Data Augmentation
- Normalize the images so each input parameter has a similar data distribution
- Flipping the image horizontally: *RandomHorizontalFlip()*
- Rotating image by up to 60 degrees: *RandomRotation()* .

The augmentation is applied using the *transform.Compose()* function of Pytorch.

In [ ]:
# data augmentation

# normalization values for pretrained resnet on Imagenet
norm_mean = (0.485, 0.456, 0.406)
norm_std = (0.229, 0.224, 0.225)

batch_size = 128
validation_batch_size = 10
test_batch_size = 10

# We compute the weights of individual classes and convert them to tensors
class_weights = estimate_weights_mfb(label)
class_weights = torch.FloatTensor(class_weights)

transform_train = transforms.Compose([
                    transforms.Resize((224,224)),
                    transforms.RandomHorizontalFlip(),
                    transforms.RandomRotation(degrees=60),
                    transforms.ToTensor(),
                    transforms.Normalize(norm_mean, norm_std),
                    ])

transform_test = transforms.Compose([
                    transforms.Resize((224,224)),
                    transforms.ToTensor(),
                    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
                    ])


## Model Architecture

The DenseNet121UNet model adapts the U-Net architecture for binary image segmentation tasks, such as detecting bone fractures in medical images. It uses a pretrained DenseNet-121 as the encoder, leveraging its densely connected convolutional layers to extract multi-scale features at several spatial resolutions. These features are progressively downsampled through layers enc1 to enc5.

The decoder then reconstructs the spatial resolution by upsampling the encoded features using transposed convolution layers (up_block), integrating them with matching encoder outputs via skip connections. This fusion helps retain fine-grained spatial details crucial for accurate segmentation.

Finally, a 1×1 convolution is used to generate a single-channel output map of raw segmentation logits, which are passed through a sigmoid activation during inference to produce the final binary segmentation mask.

In [ ]:
class DenseNet121UNet(nn.Module):
    def __init__(self, n_classes=1):
        super(DenseNet121UNet, self).__init__()

        densenet = torchvision.models.densenet121(pretrained=True)

        # Use the 'features' submodule which includes conv, bn, relu, pool, and dense blocks
        self.encoder = densenet.features

        # You’ll need to manually slice feature maps to simulate stages if you want skip connections
        # DenseNet has:
        #   features.conv0 -> features.norm0 -> features.relu0 -> features.pool0
        #   -> denseblock1 -> transition1 -> denseblock2 -> transition2 ...
        self.enc0 = nn.Sequential(
            densenet.features.conv0,
            densenet.features.norm0,
            densenet.features.relu0,
        )
        self.pool0 = densenet.features.pool0
        self.enc1 = densenet.features.denseblock1
        self.trans1 = densenet.features.transition1
        self.enc2 = densenet.features.denseblock2
        self.trans2 = densenet.features.transition2
        self.enc3 = densenet.features.denseblock3
        self.trans3 = densenet.features.transition3
        self.enc4 = densenet.features.denseblock4

        self.reduce_e3 = nn.Conv2d(1024, 512, kernel_size=1)
        self.reduce_e2 = nn.Conv2d(512, 256, kernel_size=1)
        self.reduce_e1 = nn.Conv2d(256, 128, kernel_size=1)
        self.reduce_e0 = nn.Conv2d(64, 64, kernel_size=1)  # Optional; already matches


        # Decoder
        self.up4 = self.up_block(1024, 512)
        self.up3 = self.up_block(512, 256)
        self.up2 = self.up_block(256, 128)
        self.up1 = self.up_block(128, 64)

        self.final_conv = nn.Conv2d(64, n_classes, kernel_size=1)

    def up_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        e0 = self.enc0(x)
        p0 = self.pool0(e0)
        e1 = self.enc1(p0)
        t1 = self.trans1(e1)
        e2 = self.enc2(t1)
        t2 = self.trans2(e2)
        e3 = self.enc3(t2)
        t3 = self.trans3(e3)
        e4 = self.enc4(t3)

        d4 = self.up4(e4) + self.reduce_e3(e3)
        d3 = self.up3(d4) + self.reduce_e2(e2)
        d2 = self.up2(d3) + self.reduce_e1(e1)
        d1 = self.up1(d2) + self.reduce_e0(e0)


        return self.final_conv(d1)


## Initial Model
Instantiation of the baseline model.

In [ ]:
model = DenseNet121UNet(n_classes=1).to(device)
criterion = nn.BCEWithLogitsLoss()  # binary mask loss
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

## Applying Labels and Creating Masks

The custom dataset loads the X-ray images and their respective binary fracture masks from COCO-style annotations, It then returns images and masks with transformations made due to resizing.

In [ ]:
from torch.utils.data import Dataset
from pycocotools.coco import COCO
from PIL import Image
import torchvision.transforms.functional as TF
import numpy as np
import os

class FracAtlasCocoDataset(Dataset):
    def __init__(self, images_dir, ann_path, image_ids, transform=None):
        """
        images_dir: path to folder with X-ray images
        ann_path: path to COCO annotation JSON file
        image_ids: list of image file names to include (train/val split)
        transform: torchvision transforms to apply
        """
        self.images_dir = images_dir
        self.coco = COCO(ann_path)
        self.image_ids = [
            img['id'] for img in self.coco.dataset['images']
            if img['file_name'] in image_ids
        ]
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.images_dir, img_info['file_name'])

        # Load image
        image = Image.open(img_path).convert('RGB')

        # Create empty mask
        mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)

        # Get annotations for the image
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)

        for ann in anns:
            m = self.coco.annToMask(ann)
            mask = np.maximum(mask, m)

        # Convert to PIL Image for transforms
        mask = Image.fromarray(mask)

        if self.transform:
            # Apply the same transform to both image and mask
            image = self.transform(image)
            mask = TF.resize(mask, image.shape[1:])  # resize mask to match image (if needed)
            mask = TF.to_tensor(mask)
            mask = (mask > 0).float()  # ensures binary (0 or 1)

        return image, mask

## Baseline Model Data Loading
Data is loaded to begin training the baseline model.

In [ ]:
train_csv = 'FracAtlas/Utilities/Fracture Split/train_combined.csv'
val_csv = 'FracAtlas/Utilities/Fracture Split/valid_combined.csv'
test_csv = 'FracAtlas/Utilities/Fracture Split/test_combined.csv'
ann_path = "FracAtlas/Annotations/COCO JSON/COCO_fracture_masks.json"
images_dir = "FracAtlas/images/sorted_images/Combined/all"

# Load train/val splits
train_ids = pd.read_csv(train_csv)['image_id'].tolist()
val_ids = pd.read_csv(val_csv)['image_id'].tolist()
test_ids = pd.read_csv(test_csv)['image_id'].tolist()

# Datasets
train_dataset = FracAtlasCocoDataset(images_dir, ann_path, train_ids, transform=transform_train)
val_dataset = FracAtlasCocoDataset(images_dir, ann_path, val_ids, transform=transform_test)
test_dataset = FracAtlasCocoDataset(images_dir, ann_path, test_ids, transform=transform_test)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

## Metrics Calculation Functions
Create Functions for computing metrics

In [ ]:
import torch
import numpy as np
from sklearn.metrics import average_precision_score

def compute_metrics(preds, targets, threshold=0.5, eps=1e-7):
    probs = torch.sigmoid(preds)
    preds_bin = (probs > threshold).float()

    preds_flat = preds_bin.view(-1)
    targets_flat = targets.view(-1)

    TP = (preds_flat * targets_flat).sum()
    FP = (preds_flat * (1 - targets_flat)).sum()
    FN = ((1 - preds_flat) * targets_flat).sum()
    TN = ((1 - preds_flat) * (1 - targets_flat)).sum()

    precision = TP / (TP + FP + eps)
    recall = TP / (TP + FN + eps)
    f1_score = 2 * precision * recall / (precision + recall + eps)
    iou = TP / (TP + FP + FN + eps)
    dice = 2 * TP / (2 * TP + FP + FN + eps)

    ap = average_precision_score(
        targets_flat.cpu().numpy(),
        torch.sigmoid(preds).view(-1).cpu().numpy()
    )

    return {
        'IoU': iou.item(),
        'Precision': precision.item(),
        'Recall': recall.item(),
        'F1': f1_score.item(),
        'mAP': ap,
        'Dice': dice.item()
    }

def test_model(model, test_loader, device):
    model.eval()
    all_metrics = []

    with torch.no_grad():
        for images, masks in test_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            outputs = F.interpolate(outputs, size=masks.shape[2:], mode='bilinear', align_corners=False)

            metrics = compute_metrics(outputs, masks)
            all_metrics.append(metrics)

    avg_metrics = {
        key: np.mean([m[key] for m in all_metrics]) for key in all_metrics[0]
    }

    print("=== Test Metrics ===")
    for k, v in avg_metrics.items():
        print(f"{k}: {v:.4f}")
    return avg_metrics



## Data Balance Calculation
Calculate ration of pixels are expected to be part of fractures or not part of fractures.

In [ ]:
# Estimate how imbalanced your dataset is
total_pixels = 0
fracture_pixels = 0

for _, mask in train_loader:
    fracture_pixels += mask.sum().item()
    total_pixels += mask.numel()
    break  # remove break to compute over the full dataset if you want

fracture_ratio = fracture_pixels / total_pixels
print(f"Fracture pixel ratio: {fracture_ratio:.6f}")

In [ ]:
# Compute pos_weight dynamically
pos_weight_value = 1.0 / (fracture_ratio + 1e-8)
print(f"Using pos_weight={pos_weight_value:.2f}")

# Set loss with pos_weight
pos_weight = torch.tensor([pos_weight_value]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

## More Metrics Calculation Functions

In [ ]:
def dice_score(pred, target, smooth=1.):
    pred = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    return (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)

def iou_score(pred, target, smooth=1.):
    pred = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    return (intersection + smooth) / (union + smooth)

def precision_score(pred, target, eps=1e-8):
    tp = (pred * target).sum()
    fp = (pred * (1 - target)).sum()
    return (tp + eps) / (tp + fp + eps)

def recall_score(pred, target, eps=1e-8):
    tp = (pred * target).sum()
    fn = ((1 - pred) * target).sum()
    return (tp + eps) / (tp + fn + eps)

def f1_score(pred, target, eps=1e-8):
    precision = precision_score(pred, target, eps)
    recall = recall_score(pred, target, eps)
    return (2 * precision * recall + eps) / (precision + recall + eps)

def accuracy_score(pred, target, eps=1e-8):
    correct = (pred == target).float().sum()
    total = target.numel()
    return correct / (total + eps)

def save_checkpoint(model, optimizer, epoch, path="checkpoint.pth"):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, path)

def compute_confusion_matrix(preds, targets):
    """
    Computes TP, TN, FP, FN for binary segmentation.
    Args:
        preds (Tensor): shape (B, 1, H, W)
        targets (Tensor): shape (B, 1, H, W)
    Returns:
        Dictionary with TP, TN, FP, FN
    """
    preds = preds.view(-1)
    targets = targets.view(-1)

    TP = ((preds == 1) & (targets == 1)).sum().item()
    TN = ((preds == 0) & (targets == 0)).sum().item()
    FP = ((preds == 1) & (targets == 0)).sum().item()
    FN = ((preds == 0) & (targets == 1)).sum().item()

    return {"TP": TP, "TN": TN, "FP": FP, "FN": FN}

def compute_mAP(preds, targets, thresholds=np.arange(0.5, 1.0, 0.05), eps=1e-8):
    """
    Computes mean Average Precision (mAP) across thresholds for binary segmentation.
    preds and targets should be torch tensors of shape (B, 1, H, W)
    """
    preds = preds.view(-1).cpu().numpy()
    targets = targets.view(-1).cpu().numpy()

    APs = []

    for thresh in thresholds:
        binarized = (preds >= thresh).astype(np.uint8)

        TP = np.logical_and(binarized == 1, targets == 1).sum()
        FP = np.logical_and(binarized == 1, targets == 0).sum()
        FN = np.logical_and(binarized == 0, targets == 1).sum()

        precision = TP / (TP + FP + eps)
        recall = TP / (TP + FN + eps)

        AP = precision * recall  # proxy for real AP (simplified)
        APs.append(AP)

    mAP = np.mean(APs)
    return mAP

## Training Baseline Model
Training the baseline model over 50 epochs

In [ ]:
import torch.nn.functional as F

train_losses = []
train_dices = []
train_ious = []
train_f1s = []
train_precisions = []
train_recalls = []
train_accuracies = []

val_losses = []
val_dices = []
val_ious = []
val_f1s = []
val_precisions = []
val_recalls = []
val_accuracies = []
val_maps = []

num_epochs = 50
best_val_dice = 0.0

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0
    total_f1 = 0.0
    total_precision = 0.0
    total_recall = 0.0
    total_acc = 0.0

    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        outputs = F.interpolate(outputs, size=masks.shape[2:], mode='bilinear', align_corners=False)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        with torch.no_grad():
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()
            total_dice += dice_score(preds, masks).item()
            total_iou += iou_score(preds, masks).item()
            total_f1 += f1_score(preds, masks).item()
            total_precision += precision_score(preds, masks).item()
            total_recall += recall_score(preds, masks).item()
            total_acc += accuracy_score(preds, masks).item()

    conf_matrix = {"TP": 0, "TN": 0, "FP": 0, "FN": 0}

    # Average training metrics for this epoch
    avg_train_loss = total_loss / len(train_loader)
    avg_train_dice = total_dice / len(train_loader)
    avg_train_iou = total_iou / len(train_loader)
    avg_train_f1 = total_f1 / len(train_loader)
    avg_train_precision = total_precision / len(train_loader)
    avg_train_recall = total_recall / len(train_loader)
    avg_train_acc = total_acc / len(train_loader)

    # Store training metrics
    train_losses.append(avg_train_loss)
    train_dices.append(avg_train_dice)
    train_ious.append(avg_train_iou)
    train_f1s.append(avg_train_f1)
    train_precisions.append(avg_train_precision)
    train_recalls.append(avg_train_recall)
    train_accuracies.append(avg_train_acc)

    # --- VALIDATION ---
    model.eval()
    total_dice = 0.0
    total_iou = 0.0
    total_f1 = 0.0
    total_precision = 0.0
    total_recall = 0.0
    total_acc = 0.0
    total_loss_val = 0.0

    with torch.no_grad():
        all_preds = []
        all_targets = []
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            outputs = F.interpolate(outputs, size=masks.shape[2:], mode='bilinear', align_corners=False)
            loss_val = criterion(outputs, masks)
            total_loss_val += loss_val.item()
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()

            batch_matrix = compute_confusion_matrix(preds, masks)
            for k in conf_matrix:
                conf_matrix[k] += batch_matrix[k]

            total_dice += dice_score(preds, masks).item()
            total_iou += iou_score(preds, masks).item()
            total_f1 += f1_score(preds, masks).item()
            total_precision += precision_score(preds, masks).item()
            total_recall += recall_score(preds, masks).item()
            total_acc += accuracy_score(preds, masks).item()
            all_preds.append(probs.cpu())
            all_targets.append(masks.cpu())

    avg_val_loss = total_loss_val / len(val_loader)
    avg_val_dice = total_dice / len(val_loader)
    avg_val_iou = total_iou / len(val_loader)
    avg_val_f1 = total_f1 / len(val_loader)
    avg_val_precision = total_precision / len(val_loader)
    avg_val_recall = total_recall / len(val_loader)
    avg_val_acc = total_acc / len(val_loader)

    val_losses.append(avg_val_loss)
    val_dices.append(avg_val_dice)
    val_ious.append(avg_val_iou)
    val_f1s.append(avg_val_f1)
    val_precisions.append(avg_val_precision)
    val_recalls.append(avg_val_recall)
    val_accuracies.append(avg_val_acc)

    all_preds = torch.cat(all_preds, dim=0)
    all_targets = torch.cat(all_targets, dim=0)

    val_map = compute_mAP(all_preds, all_targets)
    val_maps.append(val_map)

    print(f"Epoch {epoch+1}/{num_epochs} | "
      f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | "
      f"Dice: {avg_val_dice:.4f} | IoU: {avg_val_iou:.4f} | "
      f"mAP: {val_map:.4f} | "
      f"F1: {avg_val_f1:.4f} | Precision: {avg_val_precision:.4f} | Recall: {avg_val_recall:.4f} | Acc: {avg_val_acc:.4f}")

    if avg_val_dice > best_val_dice:
        best_val_dice = avg_val_dice
        save_checkpoint(model, optimizer, epoch)
        print("Checkpoint saved!")

## Plotting Metrics

In [ ]:
import matplotlib.pyplot as plt

epochs = list(range(1, num_epochs + 1))

plt.figure(figsize=(18, 12))

# --- Loss Curve ---
plt.subplot(2, 3, 1)
plt.plot(epochs, train_losses, 'o-', color='orange', label='Train')
plt.plot(epochs, val_losses, 'o-', color='blue', label='Validation')
plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()

# --- Dice Curve ---
plt.subplot(2, 3, 2)
plt.plot(epochs, train_dices, 'o-', color='orange', label='Train')
plt.plot(epochs, val_dices, 'o-', color='blue', label='Validation')
plt.title("Dice Score")
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.grid(True)
plt.legend()

# --- IoU Curve ---
plt.subplot(2, 3, 3)
plt.plot(epochs, train_ious, 'o-', color='orange', label='Train')
plt.plot(epochs, val_ious, 'o-', color='blue', label='Validation')
plt.title("IoU Score")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.grid(True)
plt.legend()

# --- F1 Curve ---
plt.subplot(2, 3, 4)
plt.plot(epochs, train_f1s, 'o-', color='orange', label='Train')
plt.plot(epochs, val_f1s, 'o-', color='blue', label='Validation')
plt.title("F1 Scores")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.grid(True)
plt.legend()

# --- Precision Curve ---
plt.subplot(2, 3, 5)
plt.plot(epochs, train_precisions, 'o-', color='orange', label='Train')
plt.plot(epochs, val_precisions, 'o-', color='blue', label='Validation')
plt.title("Precision")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.grid(True)
plt.legend()

# --- Recall Curve ---
plt.subplot(2, 3, 6)
plt.plot(epochs, train_recalls, 'o-', color='orange', label='Train')
plt.plot(epochs, val_recalls, 'o-', color='blue', label='Validation')
plt.title("Recall")
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

## Prediction Function
A function to generate a given image with the predicted location of the bounding box.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF

def show_prediction_with_box(model, dataset, index=0, threshold=0.5):
    model.eval()
    with torch.no_grad():
        # Get image and corresponding image_id
        image, _ = dataset[index]
        image_id = dataset.image_ids[index]
        image_info = dataset.coco.loadImgs(image_id)[0]
        filename = image_info['file_name']

        print(f"🖼️ Image ID: {filename}")

        input_tensor = image.unsqueeze(0).to(device)

        # Run model
        output = model(input_tensor)
        output = F.interpolate(output, size=image.shape[1:], mode='bilinear', align_corners=False)
        probs = torch.sigmoid(output)
        pred_mask = (probs > threshold).float().cpu().numpy()[0, 0]

        # Convert to bounding boxes using OpenCV
        mask_uint8 = (pred_mask * 255).astype(np.uint8)
        contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # Original image as NumPy for plotting
        img_np = image.permute(1, 2, 0).cpu().numpy()
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())  # Normalize for display
        img_disp = (img_np * 255).astype(np.uint8).copy()

        # Draw bounding boxes
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            cv2.rectangle(img_disp, (x, y), (x + w, y + h), (255, 0, 0), 2)

        # Show image and predicted mask
        fig, ax = plt.subplots(1, 2, figsize=(10, 5))
        ax[0].imshow(img_disp)
        ax[0].set_title("Predicted Bounding Box")
        ax[0].axis("off")

        ax[1].imshow(pred_mask, cmap='gray')
        ax[1].set_title("Predicted Mask")
        ax[1].axis("off")
        plt.show()

## Get 2 predictions

In [ ]:
# Show bounding box prediction on test image
show_prediction_with_box(model, val_dataset, index=11)
show_prediction_with_box(model, val_dataset, index=10)

## Ground Truth Function
A function to show the ground truth bounding box overlayed on the image

In [ ]:
import matplotlib.pyplot as plt
import cv2
from PIL import Image
def show_coco_box_from_original_image(dataset, index=0, image_root="FracAtlas/images/sorted_images/Combined/all"):

    image_id = dataset.image_ids[index]
    image_info = dataset.coco.loadImgs(image_id)[0]
    filename = image_info['file_name']
    filepath = os.path.join(image_root, filename)

    print(f"🖼️ Raw image with COCO boxes: {filename}")

    # Load original image
    img = cv2.imread(filepath)
    if img is None:
        print("🚨 Failed to load image.")
        return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Get annotations and draw boxes
    ann_ids = dataset.coco.getAnnIds(imgIds=image_id)
    anns = dataset.coco.loadAnns(ann_ids)

    for i, ann in enumerate(anns):
        x, y, w, h = map(int, ann['bbox'])  # COCO box format
        cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)  # 🔵 Green GT box
        print(f"🔵 GT Box {i+1}: x={x}, y={y}, w={w}, h={h}")

    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.title("🔵 COCO GT Boxes on Original Image")
    plt.axis("off")
    plt.show()

## Get the same 2 ground truth images

In [ ]:
show_coco_box_from_original_image(val_dataset, index=10)
show_coco_box_from_original_image(val_dataset, index=11)

## Metric Calculation

In [ ]:
test_metrics = test_model(model, test_loader, device)


## Reload Data for Oversampled Model

In [ ]:
# Create a new train .csv file and apply oversampling

# Load the CSVs
fracture_df = pd.read_csv('FracAtlas/Utilities/Fracture Split/train.csv')        # fracture images only
nonfracture_df = pd.read_csv('FracAtlas/Utilities/Fracture Split/nf_train.csv')  # non-fracture images only

# Oversample fracture images (4x duplication)
oversampled_fractures = pd.concat([fracture_df] * 4, ignore_index=True)

# Combine and shuffle
combined_df = pd.concat([oversampled_fractures, nonfracture_df], ignore_index=True)
combined_df = combined_df.sample(frac=1, random_state=42)  # shuffle rows


# Save the result
combined_df[['image_id']].to_csv('FracAtlas/Utilities/Fracture Split/train_combined.csv', index=False)

print(f"Combined dataset saved with {len(combined_df)} entries.")

In [ ]:
train_csv = 'FracAtlas/Utilities/Fracture Split/train_combined.csv'
val_csv = 'FracAtlas/Utilities/Fracture Split/valid_combined.csv'
test_csv = 'FracAtlas/Utilities/Fracture Split/test_combined.csv'
ann_path = "FracAtlas/Annotations/COCO JSON/COCO_fracture_masks.json"
images_dir = "FracAtlas/images/sorted_images/Combined/all"

# Load train/val splits
train_ids = pd.read_csv(train_csv)['image_id'].tolist()
val_ids = pd.read_csv(val_csv)['image_id'].tolist()
test_ids = pd.read_csv(test_csv)['image_id'].tolist()

# Datasets
train_dataset = FracAtlasCocoDataset(images_dir, ann_path, train_ids, transform=transform_train)
val_dataset = FracAtlasCocoDataset(images_dir, ann_path, val_ids, transform=transform_test)
test_dataset = FracAtlasCocoDataset(images_dir, ann_path, test_ids, transform=transform_test)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

## Data Balance Calculation

In [ ]:
# Estimate how imbalanced your dataset is
total_pixels = 0
fracture_pixels = 0

for _, mask in train_loader:
    fracture_pixels += mask.sum().item()
    total_pixels += mask.numel()
    break  # remove break to compute over the full dataset if you want

fracture_ratio = fracture_pixels / total_pixels
print(f"Fracture pixel ratio: {fracture_ratio:.6f}")

In [ ]:
# Compute pos_weight dynamically
pos_weight_value = 1.0 / (fracture_ratio + 1e-8)
print(f"Using pos_weight={pos_weight_value:.2f}")

# Set loss with pos_weight
pos_weight = torch.tensor([pos_weight_value]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

## Training Oversampled Model

In [ ]:
import torch.nn.functional as F

train_losses = []
train_dices = []
train_ious = []
train_f1s = []
train_precisions = []
train_recalls = []
train_accuracies = []

val_losses = []
val_dices = []
val_ious = []
val_f1s = []
val_precisions = []
val_recalls = []
val_accuracies = []
val_maps = []

num_epochs = 50
best_val_dice = 0.0

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0
    total_f1 = 0.0
    total_precision = 0.0
    total_recall = 0.0
    total_acc = 0.0

    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        outputs = F.interpolate(outputs, size=masks.shape[2:], mode='bilinear', align_corners=False)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        with torch.no_grad():
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()
            total_dice += dice_score(preds, masks).item()
            total_iou += iou_score(preds, masks).item()
            total_f1 += f1_score(preds, masks).item()
            total_precision += precision_score(preds, masks).item()
            total_recall += recall_score(preds, masks).item()
            total_acc += accuracy_score(preds, masks).item()

    conf_matrix = {"TP": 0, "TN": 0, "FP": 0, "FN": 0}

    # Average training metrics for this epoch
    avg_train_loss = total_loss / len(train_loader)
    avg_train_dice = total_dice / len(train_loader)
    avg_train_iou = total_iou / len(train_loader)
    avg_train_f1 = total_f1 / len(train_loader)
    avg_train_precision = total_precision / len(train_loader)
    avg_train_recall = total_recall / len(train_loader)
    avg_train_acc = total_acc / len(train_loader)

    # Store training metrics
    train_losses.append(avg_train_loss)
    train_dices.append(avg_train_dice)
    train_ious.append(avg_train_iou)
    train_f1s.append(avg_train_f1)
    train_precisions.append(avg_train_precision)
    train_recalls.append(avg_train_recall)
    train_accuracies.append(avg_train_acc)

    # --- VALIDATION ---
    model.eval()
    total_dice = 0.0
    total_iou = 0.0
    total_f1 = 0.0
    total_precision = 0.0
    total_recall = 0.0
    total_acc = 0.0
    total_loss_val = 0.0

    with torch.no_grad():
        all_preds = []
        all_targets = []
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            outputs = F.interpolate(outputs, size=masks.shape[2:], mode='bilinear', align_corners=False)
            loss_val = criterion(outputs, masks)
            total_loss_val += loss_val.item()
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()

            batch_matrix = compute_confusion_matrix(preds, masks)
            for k in conf_matrix:
                conf_matrix[k] += batch_matrix[k]

            total_dice += dice_score(preds, masks).item()
            total_iou += iou_score(preds, masks).item()
            total_f1 += f1_score(preds, masks).item()
            total_precision += precision_score(preds, masks).item()
            total_recall += recall_score(preds, masks).item()
            total_acc += accuracy_score(preds, masks).item()
            all_preds.append(probs.cpu())
            all_targets.append(masks.cpu())

    avg_val_loss = total_loss_val / len(val_loader)
    avg_val_dice = total_dice / len(val_loader)
    avg_val_iou = total_iou / len(val_loader)
    avg_val_f1 = total_f1 / len(val_loader)
    avg_val_precision = total_precision / len(val_loader)
    avg_val_recall = total_recall / len(val_loader)
    avg_val_acc = total_acc / len(val_loader)

    val_losses.append(avg_val_loss)
    val_dices.append(avg_val_dice)
    val_ious.append(avg_val_iou)
    val_f1s.append(avg_val_f1)
    val_precisions.append(avg_val_precision)
    val_recalls.append(avg_val_recall)
    val_accuracies.append(avg_val_acc)

    all_preds = torch.cat(all_preds, dim=0)
    all_targets = torch.cat(all_targets, dim=0)

    val_map = compute_mAP(all_preds, all_targets)
    val_maps.append(val_map)

    print(f"Epoch {epoch+1}/{num_epochs} | "
      f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | "
      f"Dice: {avg_val_dice:.4f} | IoU: {avg_val_iou:.4f} | "
      f"mAP: {val_map:.4f} | "
      f"F1: {avg_val_f1:.4f} | Precision: {avg_val_precision:.4f} | Recall: {avg_val_recall:.4f} | Acc: {avg_val_acc:.4f}")

    if avg_val_dice > best_val_dice:
        best_val_dice = avg_val_dice
        save_checkpoint(model, optimizer, epoch)
        print("Checkpoint saved!")

## Plotting metrics

In [ ]:
import matplotlib.pyplot as plt

epochs = list(range(1, num_epochs + 1))

plt.figure(figsize=(18, 12))

# --- Loss Curve ---
plt.subplot(2, 3, 1)
plt.plot(epochs, train_losses, 'o-', color='orange', label='Train')
plt.plot(epochs, val_losses, 'o-', color='blue', label='Validation')
plt.title("Oversampled Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()

# --- Dice Curve ---
plt.subplot(2, 3, 2)
plt.plot(epochs, train_dices, 'o-', color='orange', label='Train')
plt.plot(epochs, val_dices, 'o-', color='blue', label='Validation')
plt.title("Oversampled Dice Score")
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.grid(True)
plt.legend()

# --- IoU Curve ---
plt.subplot(2, 3, 3)
plt.plot(epochs, train_ious, 'o-', color='orange', label='Train')
plt.plot(epochs, val_ious, 'o-', color='blue', label='Validation')
plt.title("Oversampled IoU Score")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.grid(True)
plt.legend()

# --- F1 Curve ---
plt.subplot(2, 3, 4)
plt.plot(epochs, train_f1s, 'o-', color='orange', label='Train')
plt.plot(epochs, val_f1s, 'o-', color='blue', label='Validation')
plt.title("Oversampled F1 Scores")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.grid(True)
plt.legend()

# --- Precision Curve ---
plt.subplot(2, 3, 5)
plt.plot(epochs, train_precisions, 'o-', color='orange', label='Train')
plt.plot(epochs, val_precisions, 'o-', color='blue', label='Validation')
plt.title("Oversampled Precision")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.grid(True)
plt.legend()

# --- Recall Curve ---
plt.subplot(2, 3, 6)
plt.plot(epochs, train_recalls, 'o-', color='orange', label='Train')
plt.plot(epochs, val_recalls, 'o-', color='blue', label='Validation')
plt.title("Oversampled Recall")
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

## Get new predictions

In [ ]:
# Show bounding box prediction on test image
show_prediction_with_box(model, val_dataset, index=11)
show_prediction_with_box(model, val_dataset, index=10)

## Get ground truth
...again

In [ ]:
show_coco_box_from_original_image(val_dataset, index=10)
show_coco_box_from_original_image(val_dataset, index=11)

## Get new metrics

In [ ]:
test_metrics = test_model(model, test_loader, device)


## Reloading Data for High Threshold testing

In [ ]:
train_csv3 = 'FracAtlas/Utilities/Fracture Split/train_combined.csv'
val_csv3 = 'FracAtlas/Utilities/Fracture Split/valid_combined.csv'
test_csv3 = 'FracAtlas/Utilities/Fracture Split/test_combined.csv'
ann_path3 = "FracAtlas/Annotations/COCO JSON/COCO_fracture_masks.json"
images_dir3 = "FracAtlas/images/sorted_images/Combined/all"

# Load train/val splits
train_ids3 = pd.read_csv(train_csv3)['image_id'].tolist()
val_ids3 = pd.read_csv(val_csv3)['image_id'].tolist()
test_ids3 = pd.read_csv(test_csv3)['image_id'].tolist()

# Datasets
train_dataset3 = FracAtlasCocoDataset(images_dir3, ann_path3, train_ids3, transform=transform_train)
val_dataset3 = FracAtlasCocoDataset(images_dir3, ann_path3, val_ids3, transform=transform_test)
test_dataset3 = FracAtlasCocoDataset(images_dir3, ann_path3, test_ids3, transform=transform_test)

# DataLoaders
train_loader3 = DataLoader(train_dataset3, batch_size=8, shuffle=True)
val_loader3 = DataLoader(val_dataset3, batch_size=4, shuffle=False)
test_loader3 = DataLoader(test_dataset3, batch_size=4, shuffle=False)

## Redefine compute metrics
Using higher threshold (0.8 up from 0.5)

In [ ]:
import torch
import numpy as np
from sklearn.metrics import average_precision_score

def compute_metrics(preds, targets, threshold=0.8, eps=1e-7):
    probs = torch.sigmoid(preds)
    preds_bin = (probs > threshold).float()

    preds_flat = preds_bin.view(-1)
    targets_flat = targets.view(-1)

    TP = (preds_flat * targets_flat).sum()
    FP = (preds_flat * (1 - targets_flat)).sum()
    FN = ((1 - preds_flat) * targets_flat).sum()
    TN = ((1 - preds_flat) * (1 - targets_flat)).sum()

    precision = TP / (TP + FP + eps)
    recall = TP / (TP + FN + eps)
    f1_score = 2 * precision * recall / (precision + recall + eps)
    iou = TP / (TP + FP + FN + eps)
    dice = 2 * TP / (2 * TP + FP + FN + eps)

    ap = average_precision_score(
        targets_flat.cpu().numpy(),
        torch.sigmoid(preds).view(-1).cpu().numpy()
    )

    return {
        'IoU': iou.item(),
        'Precision': precision.item(),
        'Recall': recall.item(),
        'F1': f1_score.item(),
        'mAP': ap,
        'Dice': dice.item()
    }

def test_model(model, test_loader, device):
    model.eval()
    all_metrics = []

    with torch.no_grad():
        for images, masks in test_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            outputs = F.interpolate(outputs, size=masks.shape[2:], mode='bilinear', align_corners=False)

            metrics = compute_metrics(outputs, masks)
            all_metrics.append(metrics)

    avg_metrics = {
        key: np.mean([m[key] for m in all_metrics]) for key in all_metrics[0]
    }

    print("=== Test Metrics ===")
    for k, v in avg_metrics.items():
        print(f"{k}: {v:.4f}")
    return avg_metrics



## Calculate pixel imbalance

In [ ]:
# Estimate how imbalanced your dataset is
total_pixels = 0
fracture_pixels = 0

for _, mask in train_loader:
    fracture_pixels += mask.sum().item()
    total_pixels += mask.numel()
    break  # remove break to compute over the full dataset if you want

fracture_ratio3 = fracture_pixels / total_pixels
print(f"Fracture pixel ratio: {fracture_ratio:.6f}")

In [ ]:
# Compute pos_weight dynamically
pos_weight_value3 = 1.0 / (fracture_ratio3 + 1e-8)
print(f"Using pos_weight={pos_weight_value3:.2f}")

# Set loss with pos_weight
pos_weight3 = torch.tensor([pos_weight_value3]).to(device)
criterion3 = nn.BCEWithLogitsLoss(pos_weight=pos_weight3)

## Training with new higher threshold

In [ ]:
import torch.nn.functional as F

train_losses3 = []
train_dices3 = []
train_ious3 = []
train_f1s3 = []
train_precisions3 = []
train_recalls3 = []
train_accuracies3 = []

val_losses3 = []
val_dices3 = []
val_ious3 = []
val_f1s3 = []
val_precisions3 = []
val_recalls3 = []
val_accuracies3 = []
val_maps3 = []

num_epochs3 = 50
best_val_dice3 = 0.0

for epoch in range(num_epochs3):
    model.train()
    total_loss3 = 0.0
    total_dice3 = 0.0
    total_iou3 = 0.0
    total_f13 = 0.0
    total_precision3 = 0.0
    total_recall3 = 0.0
    total_acc3 = 0.0

    for images3, masks3 in train_loader3:
        images3, masks3 = images3.to(device), masks3.to(device)

        optimizer.zero_grad()
        outputs3 = model(images3)
        outputs3 = F.interpolate(outputs3, size=masks3.shape[2:], mode='bilinear', align_corners=False)
        loss3 = criterion(outputs3, masks3)
        loss3.backward()
        optimizer.step()
        total_loss3 += loss3.item()

        with torch.no_grad():
            probs3 = torch.sigmoid(outputs3)
            preds3 = (probs3 > 0.5).float()
            total_dice3 += dice_score(preds3, masks3).item()
            total_iou3 += iou_score(preds3, masks3).item()
            total_f13 += f1_score(preds3, masks3).item()
            total_precision3 += precision_score(preds3, masks3).item()
            total_recall3 += recall_score(preds3, masks3).item()
            total_acc3 += accuracy_score(preds3, masks3).item()

    conf_matrix3 = {"TP": 0, "TN": 0, "FP": 0, "FN": 0}

    # Average training metrics for this epoch
    avg_train_loss3 = total_loss3 / len(train_loader3)
    avg_train_dice3 = total_dice3 / len(train_loader3)
    avg_train_iou3 = total_iou3 / len(train_loader3)
    avg_train_f13 = total_f13 / len(train_loader3)
    avg_train_precision3 = total_precision3 / len(train_loader3)
    avg_train_recall3 = total_recall3 / len(train_loader3)
    avg_train_acc3 = total_acc3 / len(train_loader3)

    # Store training metrics
    train_losses3.append(avg_train_loss3)
    train_dices3.append(avg_train_dice3)
    train_ious3.append(avg_train_iou3)
    train_f1s3.append(avg_train_f13)
    train_precisions3.append(avg_train_precision3)
    train_recalls3.append(avg_train_recall3)
    train_accuracies3.append(avg_train_acc3)

    # --- VALIDATION ---
    model.eval()
    total_dice3 = 0.0
    total_iou3 = 0.0
    total_f13 = 0.0
    total_precision3 = 0.0
    total_recall3 = 0.0
    total_acc3 = 0.0
    total_loss_val3 = 0.0

    with torch.no_grad():
        all_preds3 = []
        all_targets3 = []
        for images3, masks3 in val_loader3:
            images3, masks3 = images3.to(device), masks3.to(device)
            outputs3 = model(images3)
            outputs3 = F.interpolate(outputs3, size=masks3.shape[2:], mode='bilinear', align_corners=False)
            loss_val3 = criterion3(outputs3, masks3)
            total_loss_val3 += loss_val.item()
            probs3 = torch.sigmoid(outputs3)
            preds3 = (probs3 > 0.5).float()

            batch_matrix3 = compute_confusion_matrix(preds3, masks3)
            for k in conf_matrix3:
                conf_matrix3[k] += batch_matrix3[k]

            total_dice3 += dice_score(preds3, masks3).item()
            total_iou3 += iou_score(preds3, masks3).item()
            total_f13 += f1_score(preds3, masks3).item()
            total_precision3 += precision_score(preds3, masks3).item()
            total_recall3 += recall_score(preds3, masks3).item()
            total_acc3 += accuracy_score(preds3, masks3).item()
            all_preds3.append(probs3.cpu())
            all_targets3.append(masks3.cpu())

    avg_val_loss3 = total_loss_val3 / len(val_loader3)
    avg_val_dice3 = total_dice3 / len(val_loader3)
    avg_val_iou3 = total_iou3 / len(val_loader3)
    avg_val_f13 = total_f13 / len(val_loader3)
    avg_val_precision3 = total_precision3 / len(val_loader3)
    avg_val_recall3 = total_recall3 / len(val_loader3)
    avg_val_acc3 = total_acc3 / len(val_loader3)

    val_losses3.append(avg_val_loss3)
    val_dices3.append(avg_val_dice3)
    val_ious3.append(avg_val_iou3)
    val_f1s3.append(avg_val_f13)
    val_precisions3.append(avg_val_precision3)
    val_recalls3.append(avg_val_recall3)
    val_accuracies3.append(avg_val_acc3)

    all_preds3 = torch.cat(all_preds3, dim=0)
    all_targets3 = torch.cat(all_targets3, dim=0)

    val_map3 = compute_mAP(all_preds3, all_targets3)
    val_maps3.append(val_map3)

    print(f"Epoch {epoch+1}/{num_epochs3} | "
      f"Train Loss: {avg_train_loss3:.4f} | Val Loss: {avg_val_loss3:.4f} | "
      f"Dice: {avg_val_dice3:.4f} | IoU: {avg_val_iou3:.4f} | "
      f"mAP: {val_map:3.4f} | "
      f"F1: {avg_val_f13:.4f} | Precision: {avg_val_precision3:.4f} | Recall: {avg_val_recall3:.4f} | Acc: {avg_val_acc3:.4f}")

    if avg_val_dice3 > best_val_dice3:
        best_val_dice3 = avg_val_dice3
        save_checkpoint(model, optimizer, epoch)
        print("Checkpoint saved!")

## Plotting metrics

In [ ]:
import matplotlib.pyplot as plt

epochs = list(range(1, num_epochs + 1))

plt.figure(figsize=(18, 12))

# --- Loss Curve ---
plt.subplot(2, 3, 1)
plt.plot(epochs, train_losses3, 'o-', color='orange', label='Train')
plt.plot(epochs, val_losses3, 'o-', color='blue', label='Validation')
plt.title("Higher Threshold Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()

# --- Dice Curve ---
plt.subplot(2, 3, 2)
plt.plot(epochs, train_dices3, 'o-', color='orange', label='Train')
plt.plot(epochs, val_dices3, 'o-', color='blue', label='Validation')
plt.title("Higher Threshold Dice Score")
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.grid(True)
plt.legend()

# --- IoU Curve ---
plt.subplot(2, 3, 3)
plt.plot(epochs, train_ious3, 'o-', color='orange', label='Train')
plt.plot(epochs, val_ious3, 'o-', color='blue', label='Validation')
plt.title("Higher Threshold IoU Score")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.grid(True)
plt.legend()

# --- F1 Curve ---
plt.subplot(2, 3, 4)
plt.plot(epochs, train_f1s3, 'o-', color='orange', label='Train')
plt.plot(epochs, val_f1s3, 'o-', color='blue', label='Validation')
plt.title("Higher Threshold F1 Scores")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.grid(True)
plt.legend()

# --- Precision Curve ---
plt.subplot(2, 3, 5)
plt.plot(epochs, train_precisions3, 'o-', color='orange', label='Train')
plt.plot(epochs, val_precisions3, 'o-', color='blue', label='Validation')
plt.title("Higher Threshold Precision")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.grid(True)
plt.legend()

# --- Recall Curve ---
plt.subplot(2, 3, 6)
plt.plot(epochs, train_recalls3, 'o-', color='orange', label='Train')
plt.plot(epochs, val_recalls3, 'o-', color='blue', label='Validation')
plt.title("Higher Threshold Recall")
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

## Redefine prediction function
Using higher threshold (0.8 up from 0.5)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF

def show_prediction_with_box(model, dataset, index=0, threshold=0.8):
    model.eval()
    with torch.no_grad():
        # Get image and corresponding image_id
        image, _ = dataset[index]
        image_id = dataset.image_ids[index]
        image_info = dataset.coco.loadImgs(image_id)[0]
        filename = image_info['file_name']

        print(f"🖼️ Image ID: {filename}")

        input_tensor = image.unsqueeze(0).to(device)

        # Run model
        output = model(input_tensor)
        output = F.interpolate(output, size=image.shape[1:], mode='bilinear', align_corners=False)
        probs = torch.sigmoid(output)
        pred_mask = (probs > threshold).float().cpu().numpy()[0, 0]

        # Convert to bounding boxes using OpenCV
        mask_uint8 = (pred_mask * 255).astype(np.uint8)
        contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # Original image as NumPy for plotting
        img_np = image.permute(1, 2, 0).cpu().numpy()
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())  # Normalize for display
        img_disp = (img_np * 255).astype(np.uint8).copy()

        # Draw bounding boxes
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            cv2.rectangle(img_disp, (x, y), (x + w, y + h), (255, 0, 0), 2)

        # Show image and predicted mask
        fig, ax = plt.subplots(1, 2, figsize=(10, 5))
        ax[0].imshow(img_disp)
        ax[0].set_title("Predicted Bounding Box")
        ax[0].axis("off")

        ax[1].imshow(pred_mask, cmap='gray')
        ax[1].set_title("Predicted Mask")
        ax[1].axis("off")
        plt.show()

In [ ]:
# Show bounding box prediction on test image
show_prediction_with_box(model, val_dataset3, index=11)
show_prediction_with_box(model, val_dataset3, index=10)

In [ ]:
show_coco_box_from_original_image(val_dataset3, index=10)
show_coco_box_from_original_image(val_dataset3, index=11)

In [ ]:
test_metrics = test_model(model, test_loader3, device)
